# V6 Qwen2.5-1.5B final-token router OOD experiment

This notebook asks whether a substantially larger **causal decoder** can rank
fallback-relative replacement safety better than notebook 05's masked-mean
ModernBERT.  It reuses the same pinned three-tier Qwen outcomes, deterministic
dataset-OOD split, seed, binary safety labels, class-balanced BCE, calibration,
threshold grid, latency scenario, and quality gates.

Only the router representation changes.  A separate
`Qwen/Qwen2.5-1.5B-Instruct` instance reads the prompt, its existing one-token
`<|endoftext|>` sentinel is appended, and two independent safety logits are
computed from the sentinel's final hidden state.  The router never invokes text
generation, does not observe candidate answers, and does not directly emit a
model name.  The analytical selector still chooses the fastest candidate whose
calibrated safety probability clears the frozen threshold.

Because notebook 05 seed 42 has already been inspected and motivated this
architecture, this default seed-42 run is a development comparison—not new
sealed confirmation.  Reserve untouched seeds or new task families for a final
claim.


## 1. Set up Colab

Choose a GPU runtime.  A Tesla T4 is the minimum practical target; an A100 is
strongly preferred because five epochs over 1,024-token prompts are expensive.
The notebook downloads Qwen2.5-1.5B once as the router.  It never generates new
candidate answers—the benchmark outcomes remain the pinned published records.

The Qwen router casts its final hidden state to each classifier head's dtype
before projection, so FP16/BF16 encoder output also works during inference
without autocast. For example, FP16 `[4, 4]` becomes FP32 `[4, 4]` and a
unit-weight, zero-bias head still returns `8`. This fixes the post-training
`Half and Float` error. Restart the runtime and run from setup after syncing
the corrected `src/llm_router/models/qwen_last_token_router.py` to the `poc`
branch fetched below; a notebook update alone does not update remote source.


In [ ]:
%cd /content

from pathlib import Path

REPOSITORY_URL = "https://github.com/BrunoVitti96/LLM-router.git"
REPOSITORY_BRANCH = "poc"
PROJECT_ROOT = Path("/content/LLM_Router")

!test -d {PROJECT_ROOT} || git clone --branch {REPOSITORY_BRANCH} {REPOSITORY_URL} {PROJECT_ROOT}
!git -C {PROJECT_ROOT} fetch origin {REPOSITORY_BRANCH}
!git -C {PROJECT_ROOT} switch {REPOSITORY_BRANCH}
!git -C {PROJECT_ROOT} pull --ff-only origin {REPOSITORY_BRANCH}
%cd /content/LLM_Router
%pip install -q -U ".[notebook]"

import importlib
import sys

REQUIRED_EVIDENCE_HELPER = PROJECT_ROOT / "src/llm_router/qwen_evidence.py"
REQUIRED_QWEN_ROUTER = (
    PROJECT_ROOT / "src/llm_router/models/qwen_last_token_router.py"
)
if not REQUIRED_EVIDENCE_HELPER.is_file() or not REQUIRED_QWEN_ROUTER.is_file():
    raise RuntimeError(
        "Repository/source mismatch: notebook 06 requires both the evidence "
        "helper and Qwen last-token router source. "
        f"Confirm that notebook 06 and branch {REPOSITORY_BRANCH!r} come from "
        "the same repository version, then rerun this setup cell."
    )

SOURCE_ROOT = str(PROJECT_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
for module_name in list(sys.modules):
    if module_name == "llm_router" or module_name.startswith("llm_router."):
        del sys.modules[module_name]
importlib.invalidate_caches()
llm_router = importlib.import_module("llm_router")
print(
    f"Router package ready from {Path(llm_router.__file__).resolve()} "
    f"on branch {REPOSITORY_BRANCH!r}"
)


/content
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 74 (delta 20), reused 74 (delta 20), pack-reused 0 (from 0)
Unpacking objects: 100% (74/74), 16.76 MiB | 10.98 MiB/s, done.
From https://github.com/BrunoVitti96/LLM-router
 * branch            poc        -> FETCH_HEAD
   e4340df..864ab83  poc        -> origin/poc
Already on 'poc'
Your branch is behind 'origin/poc' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/BrunoVitti96/LLM-router
 * branch            poc        -> FETCH_HEAD
Updating e4340df..864ab83
Fast-forward
 README.md                                          |    262 +-
 audits.md                                          |     50 +
 docs/COLAB_RUNBOOK.md                              |     24 +-
 .../06_train_qwen15_last_token_router_ood_v6.ipynb |   3130 +
 pyproject.toml                                     |

## 2. Freeze the v6 causal-router contract

The data and policy contract are unchanged from V5.  Qwen2.5-1.5B replaces
ModernBERT only as the prompt representation model.  Rank-4 LoRA is trained for
all five epochs, and the minimum OOD-validation safety loss checkpoint is
restored.

Memory-safe micro-batches contain one prompt.  Four micro-batches are
accumulated before each optimizer update, preserving V5's effective batch size
of four.  For example, four prompts with individual losses `0.10`, `0.20`,
`0.30`, and `0.40` contribute the averaged update loss
$(0.10+0.20+0.30+0.40)/4=0.25$.

The frozen 4 ms and conservative 20 ms overhead assumptions are retained solely
for policy comparability.  Measured Qwen-router p50/p95 are reported separately;
candidate answer generation is still analytical-only for latency.


In [ ]:
import hashlib
import json
import os
import shutil
from dataclasses import replace
from datetime import datetime, timezone
from getpass import getpass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.errors import GatedRepoError, HfHubHTTPError
from IPython.display import display
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer

from llm_router.config import DEFAULT_CONFIG
from llm_router.experiment_comparison import (
    build_setup_comparison,
    combine_threshold_searches,
)
from llm_router.hybrid_inference import (
    HybridModernBERTRouterRuntime,
    create_gradio_demo,
)
from llm_router.models.qwen_last_token_router import (
    QWEN_LAST_TOKEN_POOLING,
    QWEN_ROUTER_TEXT_PREFIX,
    QWEN_ROUTER_TEXT_SUFFIX,
    build_qwen_last_token_router,
    format_qwen_router_text,
)
from llm_router.modernbert_poc import (
    export_modernbert_hybrid_poc,
    train_modernbert_hybrid_poc,
)
from llm_router.oracle import oracle_choices, replacement_safety_targets
from llm_router.public_benchmark import (
    EconomicsScenario,
    ModelProfile,
    export_public_benchmark,
    make_complete_panel,
    run_public_benchmark,
    select_validation_policy,
    simulate_economics,
    split_benchmark,
)
from llm_router.qwen_evidence import (
    BINARY_METRIC_PRIORITY,
    audit_aligned_outcomes,
    published_binary_score,
    validate_qwen_tier_contract,
)
from llm_router.router_overhead import benchmark_modernbert_overhead
from llm_router.utils.training import seed_everything

RUN_SPECS = {
    "qwen25_v6_qwen_router_ood_seed_42": ("dataset_ood", 42),
    "qwen25_v6_qwen_router_ood_seed_43": ("dataset_ood", 43),
    "qwen25_v6_qwen_router_ood_seed_44": ("dataset_ood", 44),
    "qwen25_v6_qwen_router_random_seed_42": ("random", 42),
    "qwen25_v6_qwen_router_random_seed_43": ("random", 43),
    "qwen25_v6_qwen_router_random_seed_44": ("random", 44),
}
RUN_ID = "qwen25_v6_qwen_router_ood_seed_42"
SPLIT_MODE, SEED = RUN_SPECS[RUN_ID]

MAX_PROMPTS_PER_TASK = 300
EVIDENCE_SAMPLE_SEED = 20260821
EPOCHS = 5
MINIMUM_EPOCHS = 5
EARLY_STOPPING_PATIENCE = None
MICRO_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
STEP_LOG_EVERY = 25
MAX_INPUT_TOKENS = 1024
MEASURE_ROUTER_OVERHEAD = True
LAUNCH_INTERACTIVE_DEMO = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "Choose Runtime > Change runtime type > GPU."

CANDIDATES = {
    "Qwen2.5-1.5B": {
        "model_repo": "Qwen/Qwen2.5-1.5B-Instruct",
        "model_revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
        "parameters_billions": 1.54,
        "evidence_repo": "open-llm-leaderboard/Qwen__Qwen2.5-1.5B-Instruct-details",
        "evidence_revision": "801e825efda393dafd601fc9f9fa85646050f3b5",
        "evaluation_run": "2024-09-19T16-22-58.240552",
    },
    "Qwen2.5-3B": {
        "model_repo": "Qwen/Qwen2.5-3B-Instruct",
        "model_revision": "aa8e72537993ba99e69dfaafa59ed015b17504d1",
        "parameters_billions": 3.09,
        "evidence_repo": "open-llm-leaderboard/Qwen__Qwen2.5-3B-Instruct-details",
        "evidence_revision": "f5b005e407b8546e16de9349a82bbd40b0133abe",
        "evaluation_run": "2024-09-19T10-05-30.339220",
    },
    "Qwen2.5-7B": {
        "model_repo": "Qwen/Qwen2.5-7B-Instruct",
        "model_revision": "a09a35458c702b33eeacc393d103063234e8bc28",
        "parameters_billions": 7.61,
        "evidence_repo": "open-llm-leaderboard/Qwen__Qwen2.5-7B-Instruct-details",
        "evidence_revision": "98dea3336d741d4433976eeaf9e5125f39c45d5b",
        "evaluation_run": "2024-09-19T16-21-01.446061",
    },
}
validate_qwen_tier_contract(CANDIDATES)
SELECTED_MODELS = tuple(CANDIDATES)
EXCLUDED_OVERLAPPING_TASKS = {
    "leaderboard_gpqa_main",
    "leaderboard_gpqa_extended",
}

HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    from google.colab.userdata import (
        NotebookAccessError,
        SecretNotFoundError,
        TimeoutException,
    )

    if not HF_TOKEN:
        HF_TOKEN = userdata.get("HF_TOKEN")
except ImportError:
    pass
except (SecretNotFoundError, NotebookAccessError, TimeoutException) as secret_error:
    print(
        f"Colab secret unavailable ({type(secret_error).__name__}). "
        "Using a hidden session-only token prompt instead."
    )
if not HF_TOKEN:
    HF_TOKEN = getpass(
        "Paste your Hugging Face read token (input is hidden): "
    ).strip()
if not HF_TOKEN:
    raise RuntimeError(
        "No Hugging Face token was provided. Create a read token, accept "
        "access to the three gated datasets, and rerun this cell."
    )
try:
    hf_identity = HfApi().whoami(token=HF_TOKEN)
except HfHubHTTPError as token_error:
    raise RuntimeError(
        "Hugging Face rejected HF_TOKEN. Create a valid read token, "
        "enable notebook access, and rerun this cell."
    ) from token_error
print(
    "Hugging Face token accepted for "
    f"{hf_identity.get('name') or hf_identity.get('fullname') or 'the signed-in account'}."
)

evidence_contract = {
    "source": "Hugging Face Open LLM Leaderboard per-example detail datasets",
    "candidates": CANDIDATES,
    "max_prompts_per_task": MAX_PROMPTS_PER_TASK,
    "sampling_seed": EVIDENCE_SAMPLE_SEED,
    "excluded_overlapping_tasks": sorted(EXCLUDED_OVERLAPPING_TASKS),
    "correctness_policy": {
        "source": "published task-aware per-example metric",
        "metric_priority": BINARY_METRIC_PRIORITY,
        "required_values": [0, 1],
        "quality_formula": "correct / outcomes",
    },
    "candidate_latency": "analytical BF16; published runtime is unused",
}


QWEN_ROUTER_REPO = CANDIDATES["Qwen2.5-1.5B"]["model_repo"]
QWEN_ROUTER_REVISION = CANDIDATES["Qwen2.5-1.5B"]["model_revision"]
SETUP_NAME = "qwen15_last_token_1024"
V6_QWEN_ROUTER_CONTRACT = {
    "version": "v6-qwen15-last-token",
    "development_comparison": True,
    "router_repo": QWEN_ROUTER_REPO,
    "router_revision": QWEN_ROUTER_REVISION,
    "pooling_strategy": QWEN_LAST_TOKEN_POOLING,
    "text_prefix": QWEN_ROUTER_TEXT_PREFIX,
    "text_suffix": QWEN_ROUTER_TEXT_SUFFIX,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "input_truncation_strategy": "prefix_with_last",
    "loss": "class-balanced fallback-relative safety BCE",
    "oracle_auxiliary_weight": 0.0,
    "lora_rank": 4,
    "lora_alpha": 8,
    "epochs": EPOCHS,
    "checkpoint_rule": "minimum validation safety loss",
    "micro_batch_size": MICRO_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": (
        MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
    ),
    "candidate_generation_run": False,
    "kv_cache_reuse_assumed": False,
}
assert EPOCHS == MINIMUM_EPOCHS == 5
assert EARLY_STOPPING_PATIENCE is None
assert V6_QWEN_ROUTER_CONTRACT["effective_batch_size"] == 4

# Observed V5 seed-42 values are descriptive references only.  They never enter
# training, calibration, threshold selection, or the V6 pass/fail decision.
V5_SEED42_REFERENCE = {
    "validation_loss": 0.24034847696021597,
    "validation_safe_roc_auc": 0.5690090359453666,
    "validation_unsafe_average_precision": 0.18131999714355954,
    "validation_routed_fraction": 0.10801186943620178,
    "validation_conservative_savings": 0.1154382741723079,
    "test_quality_retention_lcb": 0.9954472432372866,
    "test_quality_loss_rate_ucl": 0.028846846701213805,
    "test_guarded_dataset_retention_lcb": 0.8132531107252527,
    "test_routed_fraction": 0.35923490735206215,
    "test_conservative_savings": 0.34312370708040274,
    "test_gains": 49,
    "test_losses": 37,
}

EVIDENCE_TAG = hashlib.sha256(
    json.dumps(evidence_contract, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
RUN_CONTRACT_TAG = hashlib.sha256(
    json.dumps(
        {
            "run_id": RUN_ID,
            "evidence_tag": EVIDENCE_TAG,
            "training": V6_QWEN_ROUTER_CONTRACT,
        },
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()[:16]
OUTPUT_DIR = PROJECT_ROOT / "reports_benchmark" / RUN_ID


def log_stage(stage, **values):
    timestamp = datetime.now(timezone.utc).strftime("%H:%M:%S UTC")
    details = " | ".join(f"{key}={value}" for key, value in values.items())
    print(f"[{timestamp}] {stage}" + (f" | {details}" if details else ""))


seed_everything(SEED)
log_stage(
    "contracts frozen",
    run_id=RUN_ID,
    split=SPLIT_MODE,
    evidence_tag=EVIDENCE_TAG,
    run_contract_tag=RUN_CONTRACT_TAG,
    router="Qwen2.5-1.5B final-token classifier",
    effective_batch=V6_QWEN_ROUTER_CONTRACT["effective_batch_size"],
    epochs=EPOCHS,
    gpu=torch.cuda.get_device_name(0),
)
display(pd.DataFrame(CANDIDATES).T)
display(pd.Series(V6_QWEN_ROUTER_CONTRACT, name="V6 router contract"))


Hugging Face token accepted for Pioppo.
[20:12:45 UTC] contracts frozen | run_id=qwen25_v6_qwen_router_ood_seed_42 | split=dataset_ood | evidence_tag=7bc81cd9a9ad2731 | run_contract_tag=05522cb1fbbab22e | router=Qwen2.5-1.5B final-token classifier | effective_batch=4 | epochs=5 | gpu=Tesla T4


,model_repo,model_revision,parameters_billions,evidence_repo,evidence_revision,evaluation_run
Qwen2.5-1.5B,Qwen/Qwen2.5-1.5B-Instruct,989aa7980e4cf806f80c7fef2b1adb7bc71aa306,1.54,open-llm-leaderboard/Qwen__Qwen2.5-1.5B-Instru...,801e825efda393dafd601fc9f9fa85646050f3b5,2024-09-19T16-22-58.240552
Qwen2.5-3B,Qwen/Qwen2.5-3B-Instruct,aa8e72537993ba99e69dfaafa59ed015b17504d1,3.09,open-llm-leaderboard/Qwen__Qwen2.5-3B-Instruct...,f5b005e407b8546e16de9349a82bbd40b0133abe,2024-09-19T10-05-30.339220
Qwen2.5-7B,Qwen/Qwen2.5-7B-Instruct,a09a35458c702b33eeacc393d103063234e8bc28,7.61,open-llm-leaderboard/Qwen__Qwen2.5-7B-Instruct...,98dea3336d741d4433976eeaf9e5125f39c45d5b,2024-09-19T16-21-01.446061


,V6 router contract
version,v6-qwen15-last-token
development_comparison,True
router_repo,Qwen/Qwen2.5-1.5B-Instruct
router_revision,989aa7980e4cf806f80c7fef2b1adb7bc71aa306
pooling_strategy,last_nonpadding_token
text_prefix,
text_suffix,\n<|endoftext|>
max_input_tokens,1024
input_truncation_strategy,prefix_with_last
loss,class-balanced fallback-relative safety BCE


## 3. Download the published aligned Qwen answer datasets

Accept access to these auto-gated Hugging Face datasets before running:

- `open-llm-leaderboard/Qwen__Qwen2.5-1.5B-Instruct-details`
- `open-llm-leaderboard/Qwen__Qwen2.5-3B-Instruct-details`
- `open-llm-leaderboard/Qwen__Qwen2.5-7B-Instruct-details`

Each repository contains the same 39 evaluation tasks. V3 selects the
pinned evaluation run from each repository and excludes GPQA main and
extended because they overlap with GPQA Diamond. The remaining published
tasks include BBH, GPQA Diamond, IFEval, MATH-Hard categories, MMLU-Pro,
and MuSR.

The loader keeps the published rendered prompt, filtered response,
per-example metric, document hash, prompt hash, task name, and document
ID. This stage downloads JSONL evidence only. The later training cell loads Qwen weights solely for the router. Each selected
metric must be exactly 0 (incorrect) or 1 (correct); a fractional or
ambiguous metric stops the run instead of silently becoming a label.

IFEval publishes loose and strict instruction- and prompt-level fields.
The frozen correctness rule explicitly selects its unsuffixed
`prompt_level_strict_acc` field. For example, loose prompt accuracy 1
and strict prompt accuracy 0 produces the conservative label 0.


In [ ]:
def gated_dataset_error(repo_id, error):
    raise RuntimeError(
        f"Cannot read the gated evidence dataset {repo_id!r}. Open "
        f"https://huggingface.co/datasets/{repo_id}, sign in with the "
        "same account as HF_TOKEN, accept the dataset access conditions, "
        "and confirm that the token has Read access to gated repositories."
    ) from error

def list_evidence_files(spec):
    try:
        return HfApi().list_repo_files(
            spec["evidence_repo"],
            repo_type="dataset",
            revision=spec["evidence_revision"],
            token=HF_TOKEN,
        )
    except GatedRepoError as gated_error:
        gated_dataset_error(spec["evidence_repo"], gated_error)

def download_evidence_file(spec, filename):
    try:
        return hf_hub_download(
            repo_id=spec["evidence_repo"],
            filename=filename,
            repo_type="dataset",
            revision=spec["evidence_revision"],
            token=HF_TOKEN,
        )
    except GatedRepoError as gated_error:
        gated_dataset_error(spec["evidence_repo"], gated_error)

def first_string(value):
    if isinstance(value, str):
        return value
    if isinstance(value, dict):
        for nested in value.values():
            found = first_string(nested)
            if found is not None:
                return found
    if isinstance(value, (list, tuple)):
        for nested in value:
            found = first_string(nested)
            if found is not None:
                return found
    return None

def selected_sample_files(spec):
    files = list_evidence_files(spec)
    suffix = f"_{spec['evaluation_run']}.jsonl"
    selected = []
    for filename in files:
        basename = Path(filename).name
        if not basename.startswith("samples_") or not basename.endswith(suffix):
            continue
        task = basename[len("samples_") : -len(suffix)]
        if task not in EXCLUDED_OVERLAPPING_TASKS:
            selected.append((task, filename))
    return sorted(selected)

raw_rows = []
published_run_metadata = {}
task_sets = {}
for model_name, spec in CANDIDATES.items():
    task_files = selected_sample_files(spec)
    task_sets[model_name] = {task for task, _ in task_files}
    results_filename = str(
        Path(task_files[0][1]).parent
        / f"results_{spec['evaluation_run']}.json"
    )
    results_path = download_evidence_file(spec, results_filename)
    with open(results_path, encoding="utf-8") as handle:
        published_run_metadata[model_name] = json.load(handle)
    log_stage("published evidence download started", model=model_name, tasks=len(task_files))
    for task, filename in task_files:
        local_path = download_evidence_file(spec, filename)
        with open(local_path, encoding="utf-8") as handle:
            for line in handle:
                payload = json.loads(line)
                metric_name, score = published_binary_score(payload)
                prompt = first_string(payload.get("arguments"))
                if not prompt:
                    prompt = first_string(payload.get("doc"))
                if not prompt:
                    prompt = json.dumps(
                        payload.get("doc"), ensure_ascii=False, sort_keys=True
                    )
                response = first_string(payload.get("filtered_resps"))
                if response is None:
                    response = first_string(payload.get("resps")) or ""
                document_identity = str(payload.get("doc_hash") or "")
                if not document_identity:
                    document_identity = hashlib.sha256(
                        json.dumps(
                            payload.get("doc"),
                            ensure_ascii=False,
                            sort_keys=True,
                        ).encode("utf-8")
                    ).hexdigest()
                raw_rows.append(
                    {
                        "key": f"{task}::{payload['doc_id']}",
                        "task": task,
                        "doc_id": str(payload["doc_id"]),
                        "doc_hash": document_identity,
                        "prompt_hash": str(payload.get("prompt_hash", "")),
                        "target_hash": str(payload.get("target_hash", "")),
                        "prompt": prompt,
                        "prediction": response,
                        "target": json.dumps(
                            payload.get("target"), ensure_ascii=False, sort_keys=True
                        ),
                        "score": score,
                        "published_metric": metric_name,
                        "model": model_name,
                        "source_file": filename,
                        "evidence_repo": spec["evidence_repo"],
                        "evidence_revision": spec["evidence_revision"],
                    }
                )
    log_stage("published evidence loaded", model=model_name, rows=len(raw_rows))

assert len({frozenset(tasks) for tasks in task_sets.values()}) == 1, (
    "The pinned candidate evidence repositories do not expose identical task sets."
)
raw_records = pd.DataFrame(raw_rows)
assert raw_records.groupby(["key", "model"]).size().eq(1).all()
display(
    pd.DataFrame(
        {"model": task_sets.keys(), "published_tasks": map(len, task_sets.values())}
    )
)


results_2024-09-19T16-22-58.240552.json: 0.00B [00:00, ?B/s]

[20:12:45 UTC] published evidence download started | model=Qwen2.5-1.5B | tasks=37


(…)essions_2024-09-19T16-22-58.240552.jsonl: 0.00B [00:00, ?B/s]

(…)dgement_2024-09-19T16-22-58.240552.jsonl: 0.00B [00:00, ?B/s]

(…)tanding_2024-09-19T16-22-58.240552.jsonl: 0.00B [00:00, ?B/s]

...

(…)steries_2024-09-19T16-21-01.446061.jsonl: 0.00B [00:00, ?B/s]

(…)cements_2024-09-19T16-21-01.446061.jsonl: 0.00B [00:00, ?B/s]

(…)ocation_2024-09-19T16-21-01.446061.jsonl: 0.00B [00:00, ?B/s]

[20:14:01 UTC] published evidence loaded | model=Qwen2.5-7B | rows=61836


,model,published_tasks
0,Qwen2.5-1.5B,37
1,Qwen2.5-3B,37
2,Qwen2.5-7B,37


## 4. Align, sample, and audit the published outcomes

V3 retains only prompt keys present for all three models. It requires
document hashes and rendered prompts to agree across candidates. Sampling
is deterministic within each published task, and duplicate document hashes
are kept once globally so dataset-OOD splits cannot leak the same question
through two task variants.

Correctness comes from each benchmark's published task-aware grader. The
notebook does not try to re-grade heterogeneous tasks with a fragile text
comparison. It verifies that every score is binary, records it as the
boolean `is_correct`, and computes quality as `correct / outcomes`. For
example, 240 correct rows out of 300 give quality $240/300=80\%$.

The notebook downloads the pinned Qwen tokenizer only to count input and
output tokens. Tokenization is not model inference. Realized completion
length remains excluded from analytical latency.

In [ ]:
model_count_by_key = raw_records.groupby("key").model.nunique()
complete_keys = set(model_count_by_key[model_count_by_key.eq(len(CANDIDATES))].index)
aligned = raw_records.loc[raw_records.key.isin(complete_keys)].copy()
assert not aligned.empty
assert aligned.groupby("key").doc_hash.nunique().le(1).all()
prompt_variants = aligned.groupby("key").prompt.nunique()
if not prompt_variants.le(1).all():
    raise ValueError(
        f"{int((prompt_variants > 1).sum())} keys have model-dependent prompts."
    )

canonical = (
    aligned.sort_values(["key", "model"])
    .drop_duplicates("key")
    [["key", "task", "doc_hash", "prompt"]]
)
selected_keys = []
seen_doc_hashes = set()
for task_index, (task, task_rows) in enumerate(
    canonical.groupby("task", sort=True), start=1
):
    task_rows = task_rows.loc[~task_rows.doc_hash.isin(seen_doc_hashes)]
    count = min(MAX_PROMPTS_PER_TASK, len(task_rows))
    rng = np.random.default_rng(EVIDENCE_SAMPLE_SEED + task_index)
    positions = np.sort(rng.choice(len(task_rows), size=count, replace=False))
    selected = task_rows.iloc[positions]
    selected_keys.extend(selected.key)
    seen_doc_hashes.update(selected.doc_hash)

records = aligned.loc[aligned.key.isin(selected_keys)].copy()
records["example_id"] = records.key
records["dataset"] = records.task.str.removeprefix("leaderboard_")
records["source_split"] = "published_evaluation"
records["recorded_cost"] = 0.0

tokenizer = AutoTokenizer.from_pretrained(
    CANDIDATES["Qwen2.5-7B"]["model_repo"],
    revision=CANDIDATES["Qwen2.5-7B"]["model_revision"],
    use_fast=True,
)
prompt_token_counts = {
    key: len(tokenizer.encode(prompt, add_special_tokens=False))
    for key, prompt in records.drop_duplicates("key")[["key", "prompt"]].itertuples(
        index=False, name=None
    )
}
records["prompt_tokens"] = records.key.map(prompt_token_counts).astype(int)
records["completion_tokens"] = [
    len(tokenizer.encode(str(value), add_special_tokens=False))
    for value in records.prediction
]
del tokenizer

expected_rows = len(selected_keys) * len(CANDIDATES)
assert len(records) == expected_rows
records, quality_audit = audit_aligned_outcomes(records, SELECTED_MODELS)
assert records.groupby("example_id").model.nunique().eq(len(CANDIDATES)).all()
assert records.score.eq(records.is_correct.astype(float)).all()
log_stage(
    "published evidence aligned",
    prompts=len(selected_keys),
    outcomes=len(records),
    tasks=records.dataset.nunique(),
    qwen_weights_loaded=False,
)
display(
    records.pivot_table(
        index="dataset", columns="model", values="score", aggfunc="mean"
    ).style.format("{:.1%}")
)
display(quality_audit)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[20:14:29 UTC] published evidence aligned | prompts=8632 | outcomes=25896 | tasks=37 | qwen_weights_loaded=False


model,Qwen2.5-1.5B,Qwen2.5-3B,Qwen2.5-7B
dataset,,,
bbh_boolean_expressions,83.6%,76.8%,84.4%
bbh_causal_judgement,57.8%,62.6%,52.9%
bbh_date_understanding,41.2%,45.6%,56.4%
bbh_disambiguation_qa,47.2%,54.8%,62.8%
bbh_formal_fallacies,55.6%,50.0%,59.6%
bbh_geometric_shapes,19.6%,39.2%,51.6%
bbh_hyperbaton,66.0%,63.6%,52.0%
bbh_logical_deduction_five_objects,28.8%,41.2%,50.4%
bbh_logical_deduction_seven_objects,23.2%,40.4%,47.2%


,task,model,published_metric,outcomes,correct,incorrect,quality
0,leaderboard_bbh_boolean_expressions,Qwen2.5-1.5B,acc_norm,250,209,41,0.836000
1,leaderboard_bbh_boolean_expressions,Qwen2.5-3B,acc_norm,250,192,58,0.768000
2,leaderboard_bbh_boolean_expressions,Qwen2.5-7B,acc_norm,250,211,39,0.844000
3,leaderboard_bbh_causal_judgement,Qwen2.5-1.5B,acc_norm,187,108,79,0.577540
4,leaderboard_bbh_causal_judgement,Qwen2.5-3B,acc_norm,187,117,70,0.625668
...,...,...,...,...,...,...,...
106,leaderboard_musr_object_placements,Qwen2.5-3B,acc_norm,256,60,196,0.234375
107,leaderboard_musr_object_placements,Qwen2.5-7B,acc_norm,256,63,193,0.246094
108,leaderboard_musr_team_allocation,Qwen2.5-1.5B,acc_norm,250,97,153,0.388000
109,leaderboard_musr_team_allocation,Qwen2.5-3B,acc_norm,250,114,136,0.456000


## 5. Attach analytical latency and audit leakage

The panel rejects missing prompt/model pairs and duplicate content crossing
splits. Candidate latency is computed from parameter count, BF16 weight
precision, prompt size, expected output length, and dated hardware
assumptions. Published runtime and realized completion length are excluded.

For active parameters $P$, effective compute $F$, bandwidth $B$, and
precision $b$, the core terms are:

$$t_{compute/token}=\frac{2P}{F},\qquad
t_{memory/pass}=\frac{Pb/8}{B}.$$

The code multiplies every recorded completion length by 100 and requires
analytical latency to remain identical. That is the leakage check.

In [ ]:
analytical_records = records.assign(
    formatted_prompt=records.prompt,
    ground_truth=records.target,
)
profiles = tuple(
    ModelProfile(
        name=name,
        input_price_per_million=0.0,
        output_price_per_million=0.0,
        parameters_billions=spec["parameters_billions"],
        active_parameters_billions=spec["parameters_billions"],
        architecture="autoregressive",
        weight_bits=16,
    )
    for name, spec in CANDIDATES.items()
)
scenario = EconomicsScenario(
    name="qwen25-three-tier-analytical-latency-poc",
    as_of="2026-08-21",
    profiles=profiles,
    latency_method="analytical",
    effective_tflops=60.0,
    memory_bandwidth_gbps=900.0,
    fixed_model_overhead_s=0.015,
    router_overhead_s=0.004,
    output_base_tokens=24.0,
    output_tokens_per_prompt_token=0.20,
    output_min_tokens=16,
    output_max_tokens=256,
    notes="Published Qwen answer evidence; analytical BF16 candidate latency only.",
)
simulated = simulate_economics(analytical_records, scenario)
panel = make_complete_panel(simulated, models=SELECTED_MODELS)

counterfactual = analytical_records.copy()
counterfactual["completion_tokens"] *= 100
counterfactual_simulated = simulate_economics(counterfactual, scenario)
assert np.allclose(
    simulated.simulated_latency_s,
    counterfactual_simulated.simulated_latency_s,
), "Leakage check failed: completion length changed analytical latency."

split = split_benchmark(panel, mode=SPLIT_MODE, seed=SEED)
split_indices = {"train": split.train, "validation": split.validation, "test": split.test}
prompt_hashes = {
    name: set(panel.examples.iloc[indices].prompt_hash)
    for name, indices in split_indices.items()
}
assert prompt_hashes["train"].isdisjoint(prompt_hashes["validation"])
assert prompt_hashes["train"].isdisjoint(prompt_hashes["test"])
assert prompt_hashes["validation"].isdisjoint(prompt_hashes["test"])
if SPLIT_MODE == "dataset_ood":
    assert set(split.train_datasets).isdisjoint(split.validation_datasets)
    assert set(split.train_datasets).isdisjoint(split.test_datasets)

quality_summary = records.groupby("model").is_correct.mean().rename("quality")
latency_summary = simulated.groupby("model").simulated_latency_s.mean().rename(
    "mean_analytical_latency_s"
)
panel_summary = pd.concat([quality_summary, latency_summary], axis=1)
panel_summary["latency_vs_7b"] = (
    panel_summary.mean_analytical_latency_s
    / panel_summary.loc["Qwen2.5-7B", "mean_analytical_latency_s"]
)
display(panel_summary.style.format("{:.2%}", subset=["quality", "latency_vs_7b"]))
display(
    pd.DataFrame(
        {
            "split": list(split_indices),
            "prompts": [len(indices) for indices in split_indices.values()],
        }
    )
)
log_stage(
    "Leakage check and split audit passed",
    complete_prompts=len(panel.examples),
    train=len(split.train),
    validation=len(split.validation),
    test=len(split.test),
)


,quality,mean_analytical_latency_s,latency_vs_7b
model,,,
Qwen2.5-1.5B,34.81%,0.506837,20.73%
Qwen2.5-3B,38.46%,1.001868,40.97%
Qwen2.5-7B,44.05%,2.445441,100.00%


,split,prompts
0,train,5274
1,validation,1685
2,test,1673


[20:14:32 UTC] Leakage check and split audit passed | complete_prompts=8632 | train=5274 | validation=1685 | test=1673


## 6. Verify validation-only outcome-oracle headroom

The outcome oracle remains a diagnostic upper bound, not a training loss. It
uses recorded outcomes to ask whether safe faster choices exist under several
analytical scenarios. If oracle savings are non-positive, no prompt-only safety
classifier can create routing opportunity.

This diagnostic does not activate the oracle head and does not change
`oracle_auxiliary_weight=0.0`.

In [ ]:
def validation_oracle_metrics(candidate_panel):
    fallback_index = int(candidate_panel.score[split.train].mean(axis=0).argmax())
    choices = oracle_choices(
        candidate_panel.score,
        candidate_panel.latency,
        fallback_index=fallback_index,
    )
    indices = split.validation
    rows_index = np.arange(len(indices))
    selected = choices[indices]
    fallback_quality = candidate_panel.score[indices, fallback_index].mean()
    oracle_quality = candidate_panel.score[indices][rows_index, selected].mean()
    fallback_latency = candidate_panel.latency[indices, fallback_index].mean()
    oracle_latency = candidate_panel.latency[indices][rows_index, selected].mean()
    return {
        "fallback_model": candidate_panel.models[fallback_index],
        "quality_retention": oracle_quality / max(fallback_quality, 1e-12),
        "latency_savings": 1 - oracle_latency / fallback_latency,
        "fallback_usage": np.mean(selected == fallback_index),
    }

sensitivity_settings = {
    "balanced": {},
    "compute_conservative": {"effective_tflops": 40.0},
    "bandwidth_conservative": {"memory_bandwidth_gbps": 600.0},
    "larger_fixed_overhead": {"fixed_model_overhead_s": 0.050},
    "longer_outputs": {
        "output_base_tokens": 48.0,
        "output_tokens_per_prompt_token": 0.35,
    },
}
sensitivity_rows = []
for scenario_name, overrides in sensitivity_settings.items():
    variant = replace(scenario, name=scenario_name, **overrides)
    variant_records = simulate_economics(analytical_records, variant)
    variant_panel = make_complete_panel(variant_records, models=SELECTED_MODELS)
    sensitivity_rows.append(
        {"scenario": scenario_name, **validation_oracle_metrics(variant_panel)}
    )
sensitivity = pd.DataFrame(sensitivity_rows)
assert (sensitivity.latency_savings > 0).all(), (
    "No robust oracle headroom. Stop: this panel cannot support the routing claim."
)
display(sensitivity.style.format({"quality_retention": "{:.2%}", "latency_savings": "{:.2%}"}))


,scenario,fallback_model,quality_retention,latency_savings,fallback_usage
0,balanced,Qwen2.5-7B,127.55%,71.97%,0.091395
1,compute_conservative,Qwen2.5-7B,127.55%,72.00%,0.091395
2,bandwidth_conservative,Qwen2.5-7B,127.55%,72.11%,0.091395
3,larger_fixed_overhead,Qwen2.5-7B,127.55%,70.91%,0.091395
4,longer_outputs,Qwen2.5-7B,127.55%,71.40%,0.091395


## 7. Understand the causal last-token experiment

For candidate $m$, the Qwen router receives prompt tokens followed by the fixed
sentinel.  Under causal attention, the sentinel state $h_T$ can attend to every
earlier input token.  The deployed head is

$$s_m=w_m^\top h_T+b_m,\qquad p_m=\sigma(s_m).$$

The two candidates retain independent sigmoid outputs: both 1.5B and 3B may be
safe on one prompt.  This is deliberately different from training the language
model head to generate one mutually exclusive token such as `1.5B`.

The target remains

$$y_m(x)=\mathbf 1[Q_m(x)\ge Q_{7B}(x)],$$

and latency remains outside the neural model.  For example, probabilities 0.93
and 0.96 at a threshold of 0.90 make both alternatives eligible; the analytical
selector chooses 1.5B because it is faster.


## 8. Train the Qwen2.5-1.5B final-token router

The router uses FP16 on a T4 and BF16 on Ampere-or-newer GPUs.  Gradient
checkpointing is enabled in the model builder.  If a T4 still runs out of
memory, restart the runtime before retrying; do not silently shorten the input
or change the dataset because that would invalidate the V5 comparison.


In [ ]:
def report_step(row):
    step = int(row["step_in_epoch"])
    final_step = int(row["steps_per_epoch"])
    if step != 1 and step != final_step and step % STEP_LOG_EVERY:
        return
    print(
        f"[qwen15] epoch={int(row['epoch'])}/{int(row['epochs'])} "
        f"micro_batch={step}/{final_step} "
        f"optimizer_updates={int(row['completed_optimizer_steps'])} "
        f"loss={row['step_total_loss']:.6f} "
        f"running={row['running_train_total_loss']:.6f}",
        flush=True,
    )


def report_epoch(row):
    marker = "BEST" if row["is_best_epoch"] else "    "
    print(
        f"[qwen15] epoch={int(row['epoch'])}/{EPOCHS} {marker} "
        f"train={row['train_total_loss']:.4f} "
        f"validation={row['validation_total_loss']:.4f} "
        f"best_epoch={int(row['best_epoch_so_far'])} "
        f"seconds={row['epoch_seconds']:.1f}"
    )


router_config = replace(
    DEFAULT_CONFIG,
    seed=SEED,
    encoder_repo=QWEN_ROUTER_REPO,
    encoder_revision=QWEN_ROUTER_REVISION,
    lora_r=4,
    lora_alpha=8,
    max_input_tokens=MAX_INPUT_TOKENS,
    input_truncation_strategy="prefix_with_last",
)

torch.cuda.empty_cache()
gpu_free_bytes, gpu_total_bytes = torch.cuda.mem_get_info()
display(
    pd.Series(
        {
            "gpu": torch.cuda.get_device_name(0),
            "total_gib": gpu_total_bytes / 2**30,
            "free_gib_before_training": gpu_free_bytes / 2**30,
            "micro_batch_size": MICRO_BATCH_SIZE,
            "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
            "effective_batch_size": (
                MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
            ),
        },
        name="Qwen router preflight",
    )
)

log_stage("Qwen router training started", setup=SETUP_NAME)
try:
    selected_training = train_modernbert_hybrid_poc(
        panel,
        split,
        config=router_config,
        epochs=EPOCHS,
        batch_size=MICRO_BATCH_SIZE,
        learning_rate=1e-4,
        head_learning_rate=2e-4,
        minimum_epochs=MINIMUM_EPOCHS,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        quality_epsilon=0.0,
        safety_loss_weight=1.0,
        oracle_auxiliary_weight=0.0,
        dataset_balanced_sampling=False,
        device=DEVICE,
        progress_callback=report_epoch,
        step_progress_callback=report_step,
        router_builder=build_qwen_last_token_router,
        router_text_formatter=format_qwen_router_text,
        router_display_name="Qwen2.5-1.5B final-token router",
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    )
except RuntimeError as error:
    if "out of memory" in str(error).lower():
        raise RuntimeError(
            "Qwen router training exhausted GPU memory. Restart the runtime "
            "and use an A100; changing the frozen token budget would create a "
            "different experiment."
        ) from error
    raise

assert selected_training.epochs_completed == EPOCHS
assert not selected_training.stopped_early
assert selected_training.gradient_accumulation_steps == 4
log_stage(
    "Qwen router training finished",
    best_epoch=selected_training.best_epoch,
    truncation=f"{selected_training.input_diagnostics['truncation_rate']:.2%}",
)
display(selected_training.history)
display(selected_training.calibration_diagnostics)


,Qwen router preflight
gpu,Tesla T4
total_gib,14.563171
free_gib_before_training,14.460754
micro_batch_size,1
gradient_accumulation_steps,4
effective_batch_size,4


[20:14:40 UTC] Qwen router training started | setup=qwen15_last_token_1024


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


[qwen15] epoch=1/5 micro_batch=1/5274 optimizer_updates=0 loss=0.354137 running=0.354137
[qwen15] epoch=1/5 micro_batch=25/5274 optimizer_updates=2 loss=0.028025 running=0.565360
[qwen15] epoch=1/5 micro_batch=50/5274 optimizer_updates=8 loss=0.378980 running=0.525777
[qwen15] epoch=1/5 micro_batch=75/5274 optimizer_updates=14 loss=0.470680 running=0.557630
[qwen15] epoch=1/5 micro_batch=100/5274 optimizer_updates=21 loss=0.542417 running=0.563858
[qwen15] epoch=1/5 micro_batch=125/5274 optimizer_updates=27 loss=0.399118 running=0.553019
[qwen15] epoch=1/5 micro_batch=150/5274 optimizer_updates=33 loss=0.371150 running=0.518410
[qwen15] epoch=1/5 micro_batch=175/5274 optimizer_updates=39 loss=0.082669 running=0.497407
[qwen15] epoch=1/5 micro_batch=200/5274 optimizer_updates=46 loss=0.105816 running=0.476885
[qwen15] epoch=1/5 micro_batch=225/5274 optimizer_updates=52 loss=0.141467 running=0.442296
[qwen15] epoch=1/5 micro_batch=250/5274 optimizer_updates=58 loss=0.042094 running=0.428

RuntimeError: mat1 and mat2 must have the same dtype, but got Half and Float

## 9. Select the threshold using validation only

There is one architecture, so validation selects only its calibrated safety
threshold.  The V5 numbers displayed later are read-only references and cannot
change this policy.


In [ ]:
policy_kwargs = {
    "objective": "latency",
    "minimum_quality_retention": DEFAULT_CONFIG.minimum_quality_retention,
    "confidence": DEFAULT_CONFIG.quality_confidence,
    "validation_quality_margin": DEFAULT_CONFIG.validation_quality_margin,
    "minimum_predicted_savings": DEFAULT_CONFIG.minimum_predicted_speedup,
    "router_overhead_s": scenario.router_overhead_s,
    "conservative_router_overhead_s": DEFAULT_CONFIG.conservative_router_overhead_s,
    "minimum_macro_quality_retention": DEFAULT_CONFIG.minimum_macro_quality_retention,
    "maximum_quality_loss_rate_ucl": DEFAULT_CONFIG.maximum_quality_loss_rate_ucl,
    "minimum_routed_safety_precision_lcb": DEFAULT_CONFIG.minimum_routed_safety_precision_lcb,
    "minimum_guarded_dataset_quality_retention_lcb": (
        DEFAULT_CONFIG.minimum_guarded_dataset_quality_retention_lcb
    ),
    "minimum_guarded_dataset_prompts": DEFAULT_CONFIG.minimum_guarded_dataset_prompts,
    "minimum_consecutive_feasible_thresholds": (
        DEFAULT_CONFIG.minimum_consecutive_feasible_thresholds
    ),
    "seed": SEED,
}
frozen_validation_policy = select_validation_policy(
    panel,
    split,
    routing_probabilities=selected_training.safety_probabilities,
    router_name="qwen15_last_token_router",
    **policy_kwargs,
)
trainings = {SETUP_NAME: selected_training}
selections = {SETUP_NAME: frozen_validation_policy}
setup_comparison = build_setup_comparison(trainings, selections)
setup_comparison["max_input_tokens"] = MAX_INPUT_TOKENS
setup_comparison["input_truncation_strategy"] = "prefix_with_last"
setup_comparison["pooling_strategy"] = QWEN_LAST_TOKEN_POOLING
setup_comparison["truncation_rate"] = (
    selected_training.input_diagnostics["truncation_rate"]
)
setup_comparison["selected_for_test"] = True
setup_threshold_search = combine_threshold_searches(selections)
display(setup_comparison)
display(frozen_validation_policy.threshold_search)
log_stage(
    "POLICY FROZEN — development test comparison may now open",
    active=frozen_validation_policy.router_active,
    threshold=frozen_validation_policy.selected_threshold,
    reason="; ".join(frozen_validation_policy.failure_reasons) or "all gates passed",
)


## 10. Measure Qwen-router overhead only

This times tokenization, transfer, and one final-token classification forward
pass.  It does not generate an answer.  No KV-cache reuse is assumed when the
1.5B candidate is selected, making this a conservative standalone-router
measurement.


In [ ]:
router_overhead_benchmark = None
if MEASURE_ROUTER_OVERHEAD:
    validation_examples = panel.examples.iloc[split.validation]
    router_overhead_benchmark = benchmark_modernbert_overhead(
        selected_training.model,
        selected_training.tokenizer,
        validation_examples.prompt.to_numpy(),
        validation_examples.prompt_tokens.to_numpy(),
        device=DEVICE,
        max_input_tokens=router_config.max_input_tokens,
        input_truncation_strategy=router_config.input_truncation_strategy,
        timed_requests=min(100, len(validation_examples)),
        warmup_requests=10,
        seed=SEED,
        router_text_prefix=QWEN_ROUTER_TEXT_PREFIX,
        router_text_suffix=QWEN_ROUTER_TEXT_SUFFIX,
        router_display_name="Qwen2.5-1.5B final-token router",
    )
    measured = router_overhead_benchmark.summary["end_to_end_ms"]
    log_stage(
        "Qwen router overhead measured",
        p50_ms=f"{measured['p50']:.2f}",
        p95_ms=f"{measured['p95']:.2f}",
        candidate_generation="not run",
        policy_changed=False,
    )
    display(pd.DataFrame(router_overhead_benchmark.summary).loc[
        ["mean", "p50", "p95", "maximum"],
        ["end_to_end_ms", "model_only_ms"],
    ])


## 11. Evaluate the same seed-42 OOD test

This opens exactly the same deterministic test partition used by notebook 05,
which is useful for an apples-to-apples development comparison.  It is not a
new sealed result because V5's outcomes are already known.


In [ ]:
result = run_public_benchmark(
    panel,
    split,
    routing_probabilities=selected_training.safety_probabilities,
    router_name="qwen15_last_token_router",
    selected_setup=SETUP_NAME,
    decision_metadata={
        "router_input_tokens": selected_training.router_input_lengths,
        "router_was_truncated": selected_training.router_was_truncated,
        "router_input_max_tokens": np.full(
            len(panel.examples), router_config.max_input_tokens
        ),
        "router_input_truncation_strategy": np.full(
            len(panel.examples), router_config.input_truncation_strategy
        ),
    },
    **policy_kwargs,
)
assert result.selected_threshold == frozen_validation_policy.selected_threshold
assert result.router_active == frozen_validation_policy.router_active

router_metrics = result.summary.loc["qwen15_last_token_router"]
routed = result.decisions.selected_model.ne(result.fallback_model)
gained = result.decisions.quality_delta.gt(0)
lost = result.decisions.quality_delta.lt(0)
log_stage(
    "development test comparison complete",
    single_run_passed=result.single_run_passed,
    routed=f"{routed.mean():.2%}",
    gained=int(gained.sum()),
    lost=int(lost.sum()),
    net_answers=int(gained.sum() - lost.sum()),
    retention_lcb=f"{router_metrics.quality_retention_lcb:.2%}",
    savings_4ms=f"{router_metrics.resource_savings:.2%}",
    savings_20ms=f"{router_metrics.conservative_resource_savings:.2%}",
)
display(result.summary)
display(result.router_overhead_sensitivity)
if result.failure_reasons:
    print("Failure reasons:")
    for reason in result.failure_reasons:
        print("-", reason)

break_even_ms = float(
    result.router_overhead_sensitivity.break_even_router_overhead_ms.iloc[0]
)
overhead_rows = [
    {"source": "frozen nominal assumption", "overhead_ms": 4.0},
    {"source": "frozen conservative assumption", "overhead_ms": 20.0},
]
if router_overhead_benchmark is not None:
    measured = router_overhead_benchmark.summary["end_to_end_ms"]
    overhead_rows.extend(
        [
            {"source": "measured Qwen-router p50", "overhead_ms": measured["p50"]},
            {"source": "measured Qwen-router p95", "overhead_ms": measured["p95"]},
        ]
    )
overhead_comparison = pd.DataFrame(overhead_rows)
overhead_comparison["break_even_ms"] = break_even_ms
overhead_comparison["below_break_even"] = (
    overhead_comparison.overhead_ms < break_even_ms
)
display(overhead_comparison)


## 12. Diagnose OOD task mix and within-dataset discrimination

Global AUC can improve merely because a router recognizes task templates.  The
tables below therefore report probability ranges and per-candidate ROC-AUC
inside each test dataset whenever both safe and unsafe labels exist.  A useful
causal router should separate individual prompts, not only assign one nearly
constant score to every member of a dataset.


In [ ]:
decisions = result.decisions.assign(gained=gained, lost=lost, routed=routed)
dataset_outcomes = decisions.groupby("dataset").agg(
    prompts=("dataset", "size"),
    routed=("routed", "sum"),
    gains=("gained", "sum"),
    losses=("lost", "sum"),
    truncated=("router_was_truncated", "sum"),
)
dataset_outcomes["net_answers"] = dataset_outcomes.gains - dataset_outcomes.losses
dataset_outcomes["routed_fraction"] = (
    dataset_outcomes.routed / dataset_outcomes.prompts
)
router_dataset_metrics = result.per_dataset_metrics.loc[
    result.per_dataset_metrics.strategy.eq("qwen15_last_token_router")
].set_index("dataset")
dataset_outcomes = dataset_outcomes.join(
    router_dataset_metrics[
        [
            "quality_retention",
            "quality_retention_lcb",
            "quality_loss_rate_ucl",
            "routed_safety_precision_lcb",
            "resource_savings",
            "conservative_resource_savings",
        ]
    ]
)

test_targets = replacement_safety_targets(
    panel.score[split.test],
    selected_training.fallback_index,
    selected_training.nonfallback_indices,
    quality_epsilon=0.0,
)
within_dataset_rows = []
test_examples = panel.examples.iloc[split.test].reset_index(drop=True)
for candidate_position, candidate_index in enumerate(
    selected_training.nonfallback_indices
):
    candidate = panel.models[candidate_index]
    probabilities = selected_training.safety_probabilities[
        split.test, candidate_index
    ]
    for dataset, positions in test_examples.groupby("dataset").indices.items():
        positions = np.asarray(positions, dtype=int)
        labels = test_targets[positions, candidate_position]
        scores = probabilities[positions]
        within_dataset_rows.append(
            {
                "dataset": dataset,
                "candidate": candidate,
                "prompts": len(positions),
                "safe_prevalence": labels.mean(),
                "probability_mean": scores.mean(),
                "probability_min": scores.min(),
                "probability_max": scores.max(),
                "probability_span": scores.max() - scores.min(),
                "within_dataset_roc_auc": (
                    roc_auc_score(labels, scores)
                    if np.unique(labels).size == 2
                    else np.nan
                ),
            }
        )
within_dataset_discrimination = pd.DataFrame(within_dataset_rows)

display(dataset_outcomes.sort_values("net_answers"))
display(within_dataset_discrimination)
display(
    decisions.groupby("selected_model").agg(
        prompts=("selected_model", "size"),
        gains=("gained", "sum"),
        losses=("lost", "sum"),
    )
)


## 13. V5-versus-V6 comparison dashboard

The V5 seed-42 reference is displayed only after the V6 policy is frozen and
evaluated.  Better means stronger validation ranking and within-dataset
discrimination while still passing the conservative quality, harm, subgroup,
stability, and latency-overhead gates.


In [ ]:
best_row = selected_training.history.loc[
    selected_training.history.epoch.eq(selected_training.best_epoch)
].iloc[0]
validation_row = setup_comparison.iloc[0]
v5_v6_validation_comparison = pd.DataFrame(
    [
        {
            "router": "V5 ModernBERT prefix_1024",
            "validation_loss": V5_SEED42_REFERENCE["validation_loss"],
            "safe_roc_auc": V5_SEED42_REFERENCE["validation_safe_roc_auc"],
            "unsafe_average_precision": V5_SEED42_REFERENCE[
                "validation_unsafe_average_precision"
            ],
            "routed_fraction": V5_SEED42_REFERENCE[
                "validation_routed_fraction"
            ],
            "conservative_savings": V5_SEED42_REFERENCE[
                "validation_conservative_savings"
            ],
        },
        {
            "router": "V6 Qwen1.5B final token",
            "validation_loss": best_row.validation_safety_loss,
            "safe_roc_auc": validation_row.safe_roc_auc,
            "unsafe_average_precision": validation_row.unsafe_average_precision,
            "routed_fraction": validation_row.routed_fraction,
            "conservative_savings": validation_row.conservative_resource_savings,
        },
    ]
)
v5_v6_test_comparison = pd.DataFrame(
    [
        {
            "router": "V5 ModernBERT prefix_1024",
            "quality_retention_lcb": V5_SEED42_REFERENCE[
                "test_quality_retention_lcb"
            ],
            "quality_loss_rate_ucl": V5_SEED42_REFERENCE[
                "test_quality_loss_rate_ucl"
            ],
            "guarded_dataset_retention_lcb": V5_SEED42_REFERENCE[
                "test_guarded_dataset_retention_lcb"
            ],
            "routed_fraction": V5_SEED42_REFERENCE["test_routed_fraction"],
            "conservative_savings": V5_SEED42_REFERENCE[
                "test_conservative_savings"
            ],
            "gains": V5_SEED42_REFERENCE["test_gains"],
            "losses": V5_SEED42_REFERENCE["test_losses"],
        },
        {
            "router": "V6 Qwen1.5B final token",
            "quality_retention_lcb": router_metrics.quality_retention_lcb,
            "quality_loss_rate_ucl": router_metrics.quality_loss_rate_ucl,
            "guarded_dataset_retention_lcb": (
                router_metrics.guarded_dataset_quality_retention_lcb
            ),
            "routed_fraction": router_metrics.routed_fraction,
            "conservative_savings": router_metrics.conservative_resource_savings,
            "gains": int(gained.sum()),
            "losses": int(lost.sum()),
        },
    ]
)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes[0, 0].plot(
    selected_training.history.epoch,
    selected_training.history.train_safety_loss,
    marker="o",
    label="train",
)
axes[0, 0].plot(
    selected_training.history.epoch,
    selected_training.history.validation_safety_loss,
    marker="o",
    label="OOD validation",
)
axes[0, 0].set(
    title="1. Qwen router learning curve",
    xlabel="Epoch",
    ylabel="Class-balanced BCE",
)
axes[0, 0].legend()

axes[0, 1].bar(
    v5_v6_validation_comparison.router,
    v5_v6_validation_comparison.safe_roc_auc,
    color=["tab:blue", "tab:green"],
)
axes[0, 1].axhline(0.5, color="black", linewidth=0.8)
axes[0, 1].set(title="2. OOD-validation safety ROC-AUC", ylim=(0.45, 1.0))
axes[0, 1].tick_params(axis="x", rotation=15)

axes[1, 0].bar(
    v5_v6_test_comparison.router,
    100 * v5_v6_test_comparison.conservative_savings,
    color=["tab:blue", "tab:green"],
)
axes[1, 0].axhline(0, color="black", linewidth=0.8)
axes[1, 0].set(title="3. Test analytical savings at 20 ms", ylabel="Percent")
axes[1, 0].tick_params(axis="x", rotation=15)

plot_rows = within_dataset_discrimination.dropna(
    subset=["within_dataset_roc_auc"]
)
axes[1, 1].barh(
    plot_rows.dataset + " / " + plot_rows.candidate,
    plot_rows.within_dataset_roc_auc,
    color="tab:purple",
)
axes[1, 1].axvline(0.5, color="black", linewidth=0.8)
axes[1, 1].set(
    title="4. Test within-dataset ROC-AUC",
    xlabel="AUC (only datasets with both labels)",
    xlim=(0, 1),
)

fig.suptitle("V6 Qwen1.5B final-token router diagnostic", fontsize=16)
plt.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()
v6_diagnostic_figure = fig
display(v5_v6_validation_comparison)
display(v5_v6_test_comparison)


## 14. Export the reconstructable V6 artifact

The ZIP contains the Qwen LoRA adapter, final-token safety/oracle heads,
tokenizer, calibration, threshold frontier, prompt-level decisions,
within-dataset diagnostics, measured standalone router timing, and the unchanged
published Qwen outcome evidence.


In [ ]:
report_dir = export_public_benchmark(result, scenario, OUTPUT_DIR)
sensitivity.to_csv(report_dir / "validation_sensitivity.csv", index=False)
setup_comparison.to_csv(report_dir / "setup_comparison.csv", index=False)
setup_threshold_search.to_csv(
    report_dir / "setup_threshold_search.csv", index=False
)
records.to_parquet(report_dir / "qwen_candidate_records.parquet", index=False)
quality_audit.to_csv(report_dir / "qwen_quality_audit.csv", index=False)
panel_summary.to_csv(report_dir / "qwen_candidate_panel_summary.csv")
within_dataset_discrimination.to_csv(
    report_dir / "within_dataset_discrimination.csv", index=False
)
v5_v6_validation_comparison.to_csv(
    report_dir / "v5_v6_validation_comparison.csv", index=False
)
v5_v6_test_comparison.to_csv(
    report_dir / "v5_v6_test_comparison.csv", index=False
)
(report_dir / "qwen_evidence_contract.json").write_text(
    json.dumps(
        {**evidence_contract, "evidence_tag": EVIDENCE_TAG},
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)
(report_dir / "published_evaluation_metadata.json").write_text(
    json.dumps(published_run_metadata, indent=2, sort_keys=True),
    encoding="utf-8",
)

artifact_dir = export_modernbert_hybrid_poc(
    selected_training,
    panel.models,
    report_dir / "qwen15_last_token_router",
    selected_threshold=result.selected_threshold,
    router_active=result.router_active,
    poc_passed=result.single_run_passed,
    failure_reasons=result.failure_reasons,
    minimum_predicted_savings=DEFAULT_CONFIG.minimum_predicted_speedup,
    validation_quality_margin=DEFAULT_CONFIG.validation_quality_margin,
    minimum_macro_quality_retention=DEFAULT_CONFIG.minimum_macro_quality_retention,
    maximum_quality_loss_rate_ucl=DEFAULT_CONFIG.maximum_quality_loss_rate_ucl,
    minimum_routed_safety_precision_lcb=(
        DEFAULT_CONFIG.minimum_routed_safety_precision_lcb
    ),
    minimum_guarded_dataset_quality_retention_lcb=(
        DEFAULT_CONFIG.minimum_guarded_dataset_quality_retention_lcb
    ),
    minimum_guarded_dataset_prompts=DEFAULT_CONFIG.minimum_guarded_dataset_prompts,
    conservative_router_overhead_s=DEFAULT_CONFIG.conservative_router_overhead_s,
    minimum_consecutive_feasible_thresholds=(
        DEFAULT_CONFIG.minimum_consecutive_feasible_thresholds
    ),
    benchmark_fingerprint=result.benchmark_fingerprint,
    setup_name=SETUP_NAME,
    oracle_auxiliary_weight=0.0,
    router_overhead_benchmark=(
        router_overhead_benchmark.summary
        if router_overhead_benchmark is not None
        else None
    ),
    router_display_name="Qwen2.5-1.5B final-token safety router",
    pooling_strategy=QWEN_LAST_TOKEN_POOLING,
    router_text_prefix=QWEN_ROUTER_TEXT_PREFIX,
    router_text_suffix=QWEN_ROUTER_TEXT_SUFFIX,
    encoder_reference_compile=None,
    config=router_config,
)
overhead_comparison.to_csv(
    report_dir / "qwen_router_overhead_comparison.csv", index=False
)
dataset_outcomes.reset_index().to_csv(
    report_dir / "investor_ood_dataset_summary.csv", index=False
)
v6_diagnostic_figure.savefig(
    report_dir / "v6_qwen_router_dashboard.png", dpi=180, bbox_inches="tight"
)
(report_dir / "v6_qwen_last_token_contract.json").write_text(
    json.dumps(
        {
            **V6_QWEN_ROUTER_CONTRACT,
            "run_id": RUN_ID,
            "run_contract_tag": RUN_CONTRACT_TAG,
            "best_epoch": selected_training.best_epoch,
            "epochs_completed": selected_training.epochs_completed,
            "selected_threshold": result.selected_threshold,
            "validation_router_active": result.router_active,
            "single_run_passed": result.single_run_passed,
        },
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)
if router_overhead_benchmark is not None:
    router_overhead_benchmark.export(
        report_dir, file_stem="qwen_router_overhead"
    )

bundle_path = shutil.make_archive(str(OUTPUT_DIR.resolve()), "zip", root_dir=OUTPUT_DIR)
print("Reports:", report_dir)
print("Router artifact:", artifact_dir)
print("ZIP:", bundle_path)
try:
    from google.colab import files

    files.download(bundle_path)
except ImportError:
    pass


## 15. Interpretation checklist and interactive showcase

- Compare V6 validation ROC-AUC and unsafe average precision with V5, not only
  training loss.
- Inspect probability spans and within-dataset AUC.  A higher global AUC with
  nearly constant per-dataset probabilities is still a domain shortcut.
- Require every original quality, harm, subgroup, calibration, threshold, and
  savings gate to pass.
- Compare measured Qwen-router p95 with break-even overhead.  The frozen 4/20 ms
  assumptions are for controlled policy comparison, not a production claim.
- Remember that the Qwen router is a separate inference pass and this notebook
  assumes no KV-cache reuse if the 1.5B candidate is selected.
- Treat seed 42 as development evidence.  Confirm on untouched seeds or task
  families before changing the project recommendation.


In [ ]:
# Interactive diagnostic showcase — intentionally the final notebook cell.
demo_runtime = HybridModernBERTRouterRuntime.from_training_result(
    selected_training,
    model_names=panel.models,
    fallback_model=result.fallback_model,
    selected_threshold=result.selected_threshold,
    router_active=result.router_active,
    minimum_predicted_savings=DEFAULT_CONFIG.minimum_predicted_speedup,
    scenario=scenario,
    config=router_config,
    device=DEVICE,
    router_text_prefix=QWEN_ROUTER_TEXT_PREFIX,
    router_text_suffix=QWEN_ROUTER_TEXT_SUFFIX,
    router_display_name="Qwen2.5-1.5B final-token router",
)
demo = create_gradio_demo(demo_runtime)
if LAUNCH_INTERACTIVE_DEMO:
    demo.launch(share=True, debug=False, prevent_thread_lock=True)
demo
